# NLP Assignment 2: Word Representations (Word Vectors)
Course: Introduction to NLP
Project Topic: Fake vs. Real News Detection

Team Members:
- Muhammad Faizan (1071)
- Muhammad Asim (0840)
- Daud Arif (0833)

## Part A: Count-Based Representations

In this section we build and inspect count-based representations over our Phase 1 cleaned corpus:
1. Document-Term Count Matrix (Bag of Words) and matrix sparsity.
2. TF-IDF Matrix with high-weight words vs common words driven toward zero by low IDF.
3. N-gram models (unigrams, bigrams, trigrams) analyzing vocabulary growth and sparsity trade-offs.
4. Hand-worked numerical calculation verifying TF-IDF and bigram formulas against library output.

In [1]:
# importing all libraries for count representations and embedding models
import os
import math
import html
import unicodedata
import re
import string
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import spacy

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec, FastText

import warnings
warnings.filterwarnings('ignore')

# ensuring output directories exist
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("all libraries imported successfully")

all libraries imported successfully


In [2]:
# loading cleaned dataset from assignment 1 (phase 1)
DATA_PATH = "data/processed/cleaned_news_corpus.csv"
df = pd.read_csv(DATA_PATH)

# dropping any rows with missing text just in case
df = df.dropna(subset=['clean_text']).reset_index(drop=True)
corpus = df['clean_text'].astype(str).tolist()

print(f"total documents loaded: {len(corpus)}")
print(f"columns: {df.columns.tolist()}")
print(f"class balance: {dict(df['label'].value_counts())}")
print("\nsample clean text snippet:")
print(corpus[0][:260] + "...")

total documents loaded: 6298
columns: ['article_id', 'title', 'clean_title', 'clean_text', 'label', 'label_encoded']
class balance: {'REAL': np.int64(3171), 'FAKE': np.int64(3127)}

sample clean text snippet:
daniel greenfield shillman journalism fellow freedom center new york writer focusing radical islam final stretch election hillary rodham clinton gone war fbi word unprecedented thrown around often election ought retired still unprecedented nominee major politi...


### Task A1: Bag of Words (BoW) Matrix & Sparsity

We construct the document-term count matrix using scikit-learn's `CountVectorizer`. 
To ensure manageable memory while capturing the vocabulary, we use words appearing in at least 2 documents (`min_df=2`). We also report full unconstrained matrix statistics for reference.

In [3]:
# fitting bow matrix and calculating sparsity percentage
bow_vectorizer = CountVectorizer(min_df=2)
bow_matrix = bow_vectorizer.fit_transform(corpus)

total_entries = bow_matrix.shape[0] * bow_matrix.shape[1]
nonzero_entries = bow_matrix.nnz
zero_entries = total_entries - nonzero_entries
sparsity_pct = (zero_entries / total_entries) * 100.0

print(f"bow matrix shape: {bow_matrix.shape} (documents x terms)")
print(f"total matrix cells: {total_entries:,}")
print(f"non-zero entries: {nonzero_entries:,}")
print(f"zero entries: {zero_entries:,}")
print(f"bow sparsity: {sparsity_pct:.2f}%")

bow matrix shape: (6298, 33345) (documents x terms)
total matrix cells: 210,006,810
non-zero entries: 1,666,387
zero entries: 208,340,423
bow sparsity: 99.21%
